# Phase 6 — LoRA Adaptation
25. Implement LoRA on EyeCLIP
26. Compare frozen vs LoRA under identical 1/3/5-shot conditions

**Scope note:** per the frozen plan, only LoRA is tested here (not other PEFT methods) -- that comparison is out of scope unless a later, separately justified extension is added.

**Step 0 (required before injecting LoRA):** inspect EyeCLIP's actual module structure. Do NOT assume attention layer names match standard CLIP/ViT naming -- confirm from the loaded model itself.

In [2]:
%pip install -q peft

import sys
sys.path.insert(0, r"E:\rop\codes\EyeCLIP")  # CHANGE to your actual path

import torch, eyeclip
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CKPT_PATH = r"E:\rop\dataset\ROP-VL\eyeclip.pt"                  # CHANGE to actual path
IMAGE_ROOT = Path(r"E:\rop\dataset\ROP-VL\Image")  # CHANGE to actual path
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")     # CHANGE to actual path

model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
print("EyeCLIP loaded on", DEVICE)

Note: you may need to restart the kernel to use updated packages.


e:\.venv\Lib\site-packages\torch\jit\_serialization.py:170: FutureWarning: `torch.jit.load` is not supported in Python 3.14+ and may break. Please switch to `torch.export`.
  warnings.warn(


EyeCLIP loaded on cpu


In [3]:
# Inspect module names -- LoRA needs exact target_modules names, don't guess them.
linear_layer_names = [name for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)]
print(f"Found {len(linear_layer_names)} nn.Linear layers. Sample of names:")
for n in linear_layer_names[:20]:
    print(" ", n)
print("...")

# Look specifically for attention projection layers (common LoRA targets: q_proj/k_proj/v_proj/out_proj,
# or in_proj_weight if using nn.MultiheadAttention -- CLIP-style ViTs vary by implementation)
attn_candidates = [n for n in linear_layer_names if any(k in n.lower() for k in ["attn", "proj", "q_", "k_", "v_"])]
print(f"\nLikely attention-related layers ({len(attn_candidates)}):")
for n in attn_candidates[:20]:
    print(" ", n)

Found 106 nn.Linear layers. Sample of names:
  visual.transformer.resblocks.0.attn.out_proj
  visual.transformer.resblocks.0.mlp.c_fc
  visual.transformer.resblocks.0.mlp.c_proj
  visual.transformer.resblocks.1.attn.out_proj
  visual.transformer.resblocks.1.mlp.c_fc
  visual.transformer.resblocks.1.mlp.c_proj
  visual.transformer.resblocks.2.attn.out_proj
  visual.transformer.resblocks.2.mlp.c_fc
  visual.transformer.resblocks.2.mlp.c_proj
  visual.transformer.resblocks.3.attn.out_proj
  visual.transformer.resblocks.3.mlp.c_fc
  visual.transformer.resblocks.3.mlp.c_proj
  visual.transformer.resblocks.4.attn.out_proj
  visual.transformer.resblocks.4.mlp.c_fc
  visual.transformer.resblocks.4.mlp.c_proj
  visual.transformer.resblocks.5.attn.out_proj
  visual.transformer.resblocks.5.mlp.c_fc
  visual.transformer.resblocks.5.mlp.c_proj
  visual.transformer.resblocks.6.attn.out_proj
  visual.transformer.resblocks.6.mlp.c_fc
...

Likely attention-related layers (64):
  visual.transformer.resb

## STOP AND CONFIRM before continuing
Look at the printed layer names above and fill in `TARGET_MODULES` below with the actual suffix pattern that matches EyeCLIP's attention projections (e.g. `["q_proj", "v_proj"]` or whatever the real names show). Using the wrong names will make `peft` silently attach LoRA to zero layers -- verify `model.print_trainable_parameters()` later shows a nonzero, small trainable count before trusting any result.

In [4]:
from peft import LoraConfig, get_peft_model

TARGET_MODULES = ["out_proj", "c_fc", "c_proj"] # <-- CHANGE based on the printed names above

lora_config = LoraConfig(
    r=8,                       # low-rank dimension -- small, keeps trainable params minimal
    lora_alpha=16,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
assert trainable > 0, "LoRA attached to 0 trainable parameters -- TARGET_MODULES names don't match any layer"

trainable params: 1,474,560 || all params: 179,967,745 || trainable%: 0.8193


## Load train/val/test image data (need raw images now, not just cached embeddings -- LoRA updates the encoder itself)

In [5]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

classes_sorted = sorted(train_df["categories"].unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
NUM_CLASSES = len(classes_sorted)

class ROPImageDataset(Dataset):
    """Maps (image, categories) rows -> (preprocessed tensor, label idx). Identical to Phase 1's dataset."""
    def __init__(self, df, image_root, preprocess, class_to_idx):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.image_root / row["image"]).convert("RGB")
        x = self.preprocess(img)
        y = self.class_to_idx[row["categories"]]
        return x, y

train_ds = ROPImageDataset(train_df, IMAGE_ROOT, preprocess, class_to_idx)
val_ds   = ROPImageDataset(val_df,   IMAGE_ROOT, preprocess, class_to_idx)
test_ds  = ROPImageDataset(test_df,  IMAGE_ROOT, preprocess, class_to_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

## Train LoRA adapter + a linear head jointly (full train set)
Only LoRA's small injected matrices + the new linear head are trainable -- confirmed above (`trainable > 0`, and it should be a small fraction of EyeCLIP's total parameters). The rest of EyeCLIP stays frozen.

In [ ]:
import torch.nn as nn

IMG_DIM = 512  # confirmed in Phase 0 -- verify against your own encode_image output if changed
classifier_head = nn.Linear(IMG_DIM, NUM_CLASSES).to(DEVICE)

class_counts = train_df["categories"].value_counts().reindex(classes_sorted).values
class_weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    list(p for p in lora_model.parameters() if p.requires_grad) + list(classifier_head.parameters()),
    lr=1e-4,
)

N_EPOCHS = 10  # LoRA fine-tuning needs far fewer epochs than full training -- watch val loss, don't just run 100
best_val_f1, best_head_state, best_lora_state = -1.0, None, None

for epoch in range(N_EPOCHS):
    lora_model.train(); classifier_head.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        img_emb = lora_model.encode_image(x).float()
        logits = classifier_head(img_emb)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)

    lora_model.eval(); classifier_head.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            img_emb = lora_model.encode_image(x).float()
            preds = classifier_head(img_emb).argmax(dim=1).cpu().numpy()
            val_preds.extend(preds); val_labels.extend(y.numpy())

    _, _, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average=None, zero_division=0)
    val_macro_f1 = val_f1.mean()
    print(f"epoch {epoch}  train_loss={total_loss/len(train_ds):.4f}  val_macro_f1={val_macro_f1:.4f}")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_head_state = {k: v.clone() for k, v in classifier_head.state_dict().items()}
        best_lora_state = {k: v.clone() for k, v in lora_model.state_dict().items() if "lora_" in k}

classifier_head.load_state_dict(best_head_state)
lora_model.load_state_dict(best_lora_state, strict=False)
print(f"\nbest val macro-F1: {best_val_f1:.4f}")

## Evaluate LoRA-adapted model on test set (full-train comparison, mirrors Phase 1)

In [ ]:
lora_model.eval(); classifier_head.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        img_emb = lora_model.encode_image(x).float()
        preds = classifier_head(img_emb).argmax(dim=1).cpu().numpy()
        test_preds.extend(preds); test_labels.extend(y.numpy())

acc = accuracy_score(test_labels, test_preds)
_, _, f1_per_class, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
macro_f1 = f1_per_class.mean()

print(f"LoRA-adapted EyeCLIP -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}")
print("Reference -- Phase 1 (frozen EyeCLIP, linear): acc=0.3063  macro_f1=0.2469\n")
for c, f, s in zip(classes_sorted, f1_per_class, support):
    print(f"  {c:20s}  F1={f:.3f}  n={s}")

pd.DataFrame([
    {"representation": "frozen_eyeclip_phase1", "accuracy": 0.3063, "macro_f1": 0.2469},
    {"representation": "lora_eyeclip", "accuracy": acc, "macro_f1": macro_f1},
]).to_csv("phase6_frozen_vs_lora.csv", index=False)

## Few-shot comparison: frozen vs LoRA-adapted embeddings (plan step 26)
Extract LoRA-adapted embeddings for train/test, then reuse the exact same `build_prototypes`/`nearest_prototype_predict` and seeds as Phase 2 -- so the ONLY difference from Phase 2's numbers is whether the encoder was frozen or LoRA-adapted.

In [ ]:
@torch.no_grad()
def extract_lora_embeddings(loader):
    lora_model.eval()
    feats, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        emb = lora_model.encode_image(x).float().cpu().numpy()
        feats.append(emb); labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train_lora, y_train_lora_idx = extract_lora_embeddings(train_loader)
X_test_lora,  y_test_lora_idx  = extract_lora_embeddings(test_loader)
y_train_lora = np.array([classes_sorted[i] for i in y_train_lora_idx])
y_test_lora  = np.array([classes_sorted[i] for i in y_test_lora_idx])

np.savez("lora_embeddings.npz", X_train=X_train_lora, y_train=y_train_lora_idx,
         X_test=X_test_lora, y_test=y_test_lora_idx)

def build_prototypes(X, y, classes, k, rng):
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)

def nearest_prototype_predict(X, prototypes, classes, metric="cosine"):
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        nearest_idx = (X_n @ P_n.T).argmax(axis=1)
    else:
        dists = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)
        nearest_idx = dists.argmin(axis=1)
    return np.array([classes[i] for i in nearest_idx])

SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]

lora_fewshot_results = []
for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        prototypes = build_prototypes(X_train_lora, y_train_lora, classes_sorted, k, rng)
        preds = nearest_prototype_predict(X_test_lora, prototypes, classes_sorted, metric="cosine")
        acc_k = accuracy_score(y_test_lora, preds)
        _, _, f1_k, _ = precision_recall_fscore_support(
            y_test_lora, preds, labels=classes_sorted, average=None, zero_division=0)
        lora_fewshot_results.append({"shot": k, "seed": seed, "accuracy": acc_k, "macro_f1": f1_k.mean()})

lora_fewshot_df = pd.DataFrame(lora_fewshot_results)
lora_summary = lora_fewshot_df.groupby("shot").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
).reset_index()

# reference: Phase 2's frozen-embedding few-shot numbers, for direct side-by-side
phase2_frozen = pd.DataFrame([
    {"shot": 1, "accuracy_mean": 0.154375, "macro_f1_mean": 0.127261},
    {"shot": 3, "accuracy_mean": 0.139375, "macro_f1_mean": 0.109069},
    {"shot": 5, "accuracy_mean": 0.133125, "macro_f1_mean": 0.097823},
])
phase2_frozen["representation"] = "frozen"
lora_summary_named = lora_summary[["shot", "accuracy_mean", "macro_f1_mean"]].copy()
lora_summary_named["representation"] = "lora"

comparison = pd.concat([phase2_frozen, lora_summary_named], ignore_index=True).sort_values(["shot", "representation"])
print(comparison.to_string(index=False))
comparison.to_csv("phase6_frozen_vs_lora_fewshot.csv", index=False)

## Done — Phase 6 deliverable
Record: (1) full-train frozen vs LoRA comparison, (2) few-shot frozen vs LoRA comparison.
This is the last modeling phase per the plan -- Phase 7 is consolidating all results tables and writing the final report, not new experiments.